# Relatório Unificado 1x1 - Top Ranks

Este notebook gera um único HTML chamado `1x1_visu_scores.html`, contendo os relatórios dos ranks 1, 2 e 3 em sequência. Também exporta tabelas `.tex` e figuras `.pdf`/`.png` para a pasta `1x1_visu_scores/`, com botões de download no HTML.

In [8]:
# ============================================================
# RELATÓRIO UNIFICADO 1x1 - TOP RANKS
# HTML + DOWNLOADS + EXPORTAÇÃO PARA LaTeX
#
# Entradas:
# - creditcard.csv
# - 1x1_scores.csv
#
# Saídas:
# - 1x1_visu_scores.html
# - 1x1_visu_scores/
#     ├── tabela_metricas_ranks.tex
#     ├── tabela_matrizes_confusao_ranks.tex
#     ├── comandos_latex_exemplo.tex
#     ├── rank_1_..._boxplot.pdf/png
#     ├── rank_1_..._representacao_1d.pdf/png
#     ├── rank_1_..._spearman.pdf/png
#     ├── rank_1_..._responsabilidades_melhor_corte.pdf/png
#     ├── rank_1_..._responsabilidades_corte_medio.pdf/png
#     ├── rank_1_..._matriz_melhor_corte.pdf/png
#     ├── rank_1_..._matriz_corte_medio.pdf/png
#     ├── rank_1_..._matriz_ideal.pdf/png
#     └── ... mesmos arquivos para ranks 2 e 3
#
# Agora tudo sai em um único HTML, na ordem:
# Rank 1 -> Rank 2 -> Rank 3.
# ============================================================

import io
import re
import html
import base64
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix


# ============================================================
# CONFIGURAÇÕES
# ============================================================

ARQUIVO_DADOS = "creditcard.csv"
ARQUIVO_SCORES = "1x1_scores.csv"

PASTA_SAIDA = "."
ARQUIVO_HTML = "1x1_visu_scores.html"
PASTA_LATEX = "1x1_visu_scores"

RANKS = (1, 2, 3)

AZUL_NAO_FRAUDE = "#2563eb"
AZUL_NAO_FRAUDE_BORDA = "#1e3a8a"
AZUL_NAO_FRAUDE_CLARO = "#60a5fa"
AMARELO_FRAUDE = "#facc15"
PRETO_BORDA = "#111827"


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def sanitizar_nome(nome):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(nome))


def formatar_count(valor):
    return f"{int(valor):,}".replace(",", ".")


def formatar_float_html(valor, casas=6):
    if pd.isna(valor):
        return "-"
    try:
        return f"{float(valor):.{casas}f}".replace(".", ",")
    except Exception:
        return str(valor)


def formatar_float_latex(valor, casas=6):
    if pd.isna(valor):
        return "-"
    try:
        return f"{float(valor):.{casas}f}"
    except Exception:
        return str(valor)


def fig_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("utf-8")


def salvar_figura(fig, caminho_sem_extensao):
    caminho_sem_extensao = Path(caminho_sem_extensao)
    fig.savefig(caminho_sem_extensao.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(caminho_sem_extensao.with_suffix(".png"), dpi=300, bbox_inches="tight")


def preparar_df_latex(df):
    df_latex = df.copy()
    for col in df_latex.columns:
        if pd.api.types.is_float_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: formatar_float_latex(x, 6))
        elif pd.api.types.is_integer_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: int(x) if not pd.isna(x) else x)
    return df_latex


def salvar_tabela_latex(df, caminho, caption, label, longtable=False):
    caminho = Path(caminho)
    if df is None or df.empty:
        caminho.write_text("% Tabela vazia.\n", encoding="utf-8")
        return

    tex = preparar_df_latex(df).to_latex(
        index=False,
        escape=True,
        longtable=longtable,
        caption=caption,
        label=label
    )
    caminho.write_text(tex, encoding="utf-8")


def gerar_tabela_html(df, table_id):
    if df is None or df.empty:
        return "<p>Nenhum dado disponível.</p>"

    out = [f'<table id="{html.escape(str(table_id))}" class="data-table">']
    out.append("<thead><tr>")
    for col in df.columns:
        out.append(f"<th>{html.escape(str(col))}</th>")
    out.append("</tr></thead><tbody>")

    for _, row in df.iterrows():
        out.append("<tr>")
        for valor in row:
            out.append(f"<td>{html.escape(str(valor))}</td>")
        out.append("</tr>")

    out.append("</tbody></table>")
    return "\n".join(out)


def secao_tabela(titulo, df, table_id, nome_csv):
    tabela_html = gerar_tabela_html(df, table_id)
    return f"""
    <section class="card">
        <div class="section-header">
            <h2>{html.escape(str(titulo))}</h2>
            <button class="download-btn" onclick="baixarTabelaCSV('{table_id}', '{nome_csv}')">Baixar CSV</button>
        </div>
        <div class="table-wrapper">
            {tabela_html}
        </div>
    </section>
    """


def secao_imagem(titulo, imagem_base64, nome_download, alt_text):
    """
    Seção de imagem sem título HTML acima do gráfico.

    O próprio gráfico já possui título dentro da imagem matplotlib.
    Para evitar título duplicado, aqui fica somente o botão de download.
    """
    return f"""
    <section class="card">
        <div class="section-actions-only">
            <a class="download-btn" href="data:image/png;base64,{imagem_base64}" download="{html.escape(str(nome_download))}">Baixar PNG</a>
        </div>
        <img class="plot-img" src="data:image/png;base64,{imagem_base64}" alt="{html.escape(str(alt_text))}">
    </section>
    """


# ============================================================
# MATRIZ DE CONFUSÃO
# ============================================================

def gerar_matriz_confusao(y_real, probabilidades, threshold):
    y_pred = (probabilidades >= threshold).astype(int)
    return confusion_matrix(y_real, y_pred, labels=[0, 1])


def preparar_valores_matriz(cm):
    tn, fp, fn, tp = cm.ravel()

    total_fraudes = fn + tp
    total_nao_fraudes = tn + fp

    fn_pct = fn / total_fraudes * 100 if total_fraudes else 0
    tp_pct = tp / total_fraudes * 100 if total_fraudes else 0
    tn_pct = tn / total_nao_fraudes * 100 if total_nao_fraudes else 0
    fp_pct = fp / total_nao_fraudes * 100 if total_nao_fraudes else 0

    return {
        "fn": {"pct": fn_pct, "count": int(fn), "qualidade": 100 - fn_pct},
        "tp": {"pct": tp_pct, "count": int(tp), "qualidade": tp_pct},
        "tn": {"pct": tn_pct, "count": int(tn), "qualidade": tn_pct},
        "fp": {"pct": fp_pct, "count": int(fp), "qualidade": 100 - fp_pct},
        "raw": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    }


def preparar_valores_matriz_ideal(y_real):
    y = np.asarray(y_real).astype(int)
    n_fraude = int(np.sum(y == 1))
    n_nao_fraude = int(np.sum(y == 0))

    return {
        "fn": {"pct": 0.0, "count": 0, "qualidade": 100.0},
        "tp": {"pct": 100.0, "count": n_fraude, "qualidade": 100.0},
        "tn": {"pct": 100.0, "count": n_nao_fraude, "qualidade": 100.0},
        "fp": {"pct": 0.0, "count": 0, "qualidade": 100.0},
        "raw": {"tn": n_nao_fraude, "fp": 0, "fn": 0, "tp": n_fraude}
    }


def gerar_fig_matriz_confusao(titulo, valores):
    matriz_pct = np.array([
        [valores["fn"]["pct"], valores["tp"]["pct"]],
        [valores["tn"]["pct"], valores["fp"]["pct"]]
    ])

    matriz_count = np.array([
        [valores["fn"]["count"], valores["tp"]["count"]],
        [valores["tn"]["count"], valores["fp"]["count"]]
    ])

    fig, ax = plt.subplots(figsize=(8.5, 6.2))

    im = ax.imshow(matriz_pct, vmin=0, vmax=100, cmap="Blues")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Percentual por classe real (%)", fontweight="bold")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred Não Fraude", "Pred Fraude"], fontweight="bold")
    ax.set_yticklabels(["Real Fraude", "Real Não Fraude"], fontweight="bold")

    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=14)

    textos = [
        ["Fraude perdida", "Fraude detectada"],
        ["Não fraude correta", "Falso alerta"]
    ]

    for i in range(2):
        for j in range(2):
            ax.text(
                j, i,
                f"{matriz_pct[i, j]:.2f}%\n({formatar_count(matriz_count[i, j])})\n{textos[i][j]}",
                ha="center", va="center", color="black", fontweight="bold", fontsize=10
            )

    plt.tight_layout()
    return fig


def qualidade_cor(q):
    if q >= 95:
        return "q95"
    if q >= 85:
        return "q85"
    if q >= 70:
        return "q70"
    if q >= 50:
        return "q50"
    if q >= 30:
        return "q30"
    return "q10"


def html_matriz_visual(titulo, valores, imagem_base64, nome_download, matriz_ideal=False):
    if matriz_ideal:
        desc_fn = "Erro ideal: nenhuma fraude perdida"
        desc_tp = "Acerto ideal: fraudes detectadas"
        desc_tn = "Acerto ideal: não fraudes corretas"
        desc_fp = "Erro ideal: nenhum falso alerta"
    else:
        desc_fn = "Erro: fraude perdida"
        desc_tp = "Acerto: fraude detectada"
        desc_tn = "Acerto: não fraude"
        desc_fp = "Erro: falso alerta"

    return f"""
    <section class="matrix-card">
        <div class="section-header">
            <h2>{html.escape(str(titulo))}</h2>
            <a class="download-btn" href="data:image/png;base64,{imagem_base64}" download="{html.escape(str(nome_download))}">Baixar PNG</a>
        </div>

        <div class="matrix-area">
            <div class="matrix-wrapper">
                <div class="corner"></div>
                <div class="x-label">Pred Não Fraude</div>
                <div class="x-label">Pred Fraude</div>

                <div class="y-label">Real Fraude</div>

                <div class="cell {qualidade_cor(valores['fn']['qualidade'])}">
                    <div class="pct">{valores['fn']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['fn']['count'])})</div>
                    <div class="cell-desc">{desc_fn}</div>
                </div>

                <div class="cell {qualidade_cor(valores['tp']['qualidade'])}">
                    <div class="pct">{valores['tp']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['tp']['count'])})</div>
                    <div class="cell-desc">{desc_tp}</div>
                </div>

                <div class="y-label">Real Não Fraude</div>

                <div class="cell {qualidade_cor(valores['tn']['qualidade'])}">
                    <div class="pct">{valores['tn']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['tn']['count'])})</div>
                    <div class="cell-desc">{desc_tn}</div>
                </div>

                <div class="cell {qualidade_cor(valores['fp']['qualidade'])}">
                    <div class="pct">{valores['fp']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['fp']['count'])})</div>
                    <div class="cell-desc">{desc_fp}</div>
                </div>
            </div>

            <div class="legend">
                <div class="legend-title">Qualidade</div>
                <div class="colorbar"></div>
                <div class="legend-label-top">Melhor</div>
                <div class="legend-label-bottom">Pior</div>
            </div>
        </div>
    </section>
    """


# ============================================================
# GRÁFICOS
# ============================================================

def gerar_boxplot_1d_fig(df_plot, feature, target_name):
    dados_nao_fraude = df_plot.loc[df_plot[target_name] == 0, feature].dropna()
    dados_fraude = df_plot.loc[df_plot[target_name] == 1, feature].dropna()

    fig, ax = plt.subplots(figsize=(10, 6))

    box = ax.boxplot(
        [dados_nao_fraude, dados_fraude],
        labels=[
            f"Não Fraude ({len(dados_nao_fraude):,})".replace(",", "."),
            f"Fraude ({len(dados_fraude):,})".replace(",", ".")
        ],
        patch_artist=True,
        showfliers=True
    )

    for patch, cor in zip(box["boxes"], [AZUL_NAO_FRAUDE, AMARELO_FRAUDE]):
        patch.set_facecolor(cor)
        patch.set_alpha(0.72)
        patch.set_linewidth(2)

    for median in box["medians"]:
        median.set_color(PRETO_BORDA)
        median.set_linewidth(2.3)

    for item in box["whiskers"] + box["caps"]:
        item.set_color("#334155")
        item.set_linewidth(1.6)

    for flier in box["fliers"]:
        flier.set_marker("o")
        flier.set_markerfacecolor("#64748b")
        flier.set_markeredgecolor("#64748b")
        flier.set_alpha(0.20)
        flier.set_markersize(2.5)

    ax.set_title(f"Boxplot por Classe - {feature}", fontsize=14, fontweight="bold")
    ax.set_ylabel(feature, fontsize=14, fontweight="bold")
    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    return fig


def gerar_representacao_1d_fig(df_plot, feature, target_name):
    dados_nao_fraude = df_plot.loc[df_plot[target_name] == 0, feature].dropna()
    dados_fraude = df_plot.loc[df_plot[target_name] == 1, feature].dropna()

    rng = np.random.default_rng(42)
    y_nao_fraude = rng.normal(loc=0.0, scale=0.025, size=len(dados_nao_fraude))
    y_fraude = rng.normal(loc=1.0, scale=0.035, size=len(dados_fraude))

    fig, ax = plt.subplots(figsize=(11, 4.8))

    ax.scatter(dados_nao_fraude, y_nao_fraude, s=6, alpha=0.06, color=AZUL_NAO_FRAUDE, rasterized=True)
    ax.scatter(dados_fraude, y_fraude, s=32, alpha=0.88, color=AMARELO_FRAUDE, edgecolors=PRETO_BORDA, linewidths=0.25, rasterized=True)

    ax.set_title(f"Representação 1D com 100% dos Dados - {feature}", fontsize=14, fontweight="bold")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Não Fraude", "Fraude"], fontsize=12, fontweight="bold")
    ax.set_xlabel(feature, fontsize=14, fontweight="bold")
    ax.set_ylabel("Classe Real", fontsize=13, fontweight="bold")
    ax.grid(alpha=0.22)

    legenda_nao_fraude = plt.Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=AZUL_NAO_FRAUDE, markeredgecolor=AZUL_NAO_FRAUDE_BORDA, label=f"Não Fraude ({len(dados_nao_fraude):,})".replace(",", "."))
    legenda_fraude = plt.Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=AMARELO_FRAUDE, markeredgecolor=PRETO_BORDA, label=f"Fraude ({len(dados_fraude):,})".replace(",", "."))

    ax.legend(handles=[legenda_nao_fraude, legenda_fraude], title="Classe Real", loc="best", frameon=True)

    plt.tight_layout()
    return fig


def gerar_matriz_spearman_1d_fig(df_plot, feature, target_name):
    dados_corr = df_plot[[feature, target_name]].copy().rename(columns={target_name: "Fraude"})
    corr = dados_corr.corr(method="spearman")

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Correlação de Spearman", fontsize=11, fontweight="bold")

    labels = corr.columns.tolist()

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))

    ax.set_xticklabels(labels, fontsize=12, fontweight="bold", rotation=35, ha="right")
    ax.set_yticklabels(labels, fontsize=12, fontweight="bold")

    ax.set_title(f"Correlação de Spearman - {feature} e Fraude", fontsize=13, fontweight="bold")

    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{corr.values[i, j]:.3f}", ha="center", va="center", color="black", fontsize=13, fontweight="bold")

    plt.tight_layout()
    return fig


def calcular_pontos_corte_1d(x_grid, prob_grid, threshold):
    diff = prob_grid - threshold
    idx = np.where(np.sign(diff[:-1]) != np.sign(diff[1:]))[0]

    pontos = []

    for i in idx:
        x1 = x_grid[i]
        x2 = x_grid[i + 1]
        y1 = diff[i]
        y2 = diff[i + 1]

        if y2 == y1:
            pontos.append(x1)
        else:
            pontos.append(x1 - y1 * (x2 - x1) / (y2 - y1))

    return pontos


def gerar_responsabilidades_1d_fig(df_plot, feature, target_name, scaler, gmm, cluster_fraude, probabilidades, threshold, titulo):
    y_real_array = df_plot[target_name].astype(int).to_numpy()

    mask_nao_fraude = y_real_array == 0
    mask_fraude = y_real_array == 1

    x_original = df_plot[feature].to_numpy()

    x_min = np.min(x_original)
    x_max = np.max(x_original)
    margem = 0.05 * (x_max - x_min)

    x_grid = np.linspace(x_min - margem, x_max + margem, 1200)
    x_grid_df = pd.DataFrame({feature: x_grid})
    x_grid_scaled = scaler.transform(x_grid_df)

    prob_grid = gmm.predict_proba(x_grid_scaled)[:, cluster_fraude]
    pontos_corte = calcular_pontos_corte_1d(x_grid=x_grid, prob_grid=prob_grid, threshold=threshold)

    fig, ax = plt.subplots(figsize=(11, 6.2))

    ax.scatter(x_original[mask_nao_fraude], probabilidades[mask_nao_fraude], s=7, alpha=0.15, color=AZUL_NAO_FRAUDE_CLARO, rasterized=True)
    ax.scatter(x_original[mask_fraude], probabilidades[mask_fraude], s=34, alpha=0.90, color=AMARELO_FRAUDE, edgecolors=PRETO_BORDA, linewidths=0.25, rasterized=True)

    ax.plot(x_grid, prob_grid, color="#dc2626", linewidth=2.3, label="Responsabilidade estimada pela GMM")
    ax.axhline(y=threshold, color=PRETO_BORDA, linestyle="--", linewidth=2.0, label="Ponto de corte")

    for ponto in pontos_corte:
        ax.axvline(x=ponto, color=PRETO_BORDA, linestyle=":", linewidth=1.8, alpha=0.95)

    ax.set_title(f"{titulo} | Corte = {threshold:.6f}", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel(feature, fontsize=14, fontweight="bold")
    ax.set_ylabel("Responsabilidade GMM para fraude", fontsize=13, fontweight="bold")
    ax.set_ylim(-0.03, 1.03)
    ax.grid(alpha=0.22)

    legenda_nao_fraude = plt.Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=AZUL_NAO_FRAUDE_CLARO, markeredgecolor=AZUL_NAO_FRAUDE_BORDA, label=f"Não Fraude real ({mask_nao_fraude.sum():,})".replace(",", "."))
    legenda_fraude = plt.Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=AMARELO_FRAUDE, markeredgecolor=PRETO_BORDA, label=f"Fraude real ({mask_fraude.sum():,})".replace(",", "."))
    legenda_corte_vertical = plt.Line2D([0], [0], linestyle=":", linewidth=1.8, color=PRETO_BORDA, label="Ponto(s) onde responsabilidade = corte")

    handles, _ = ax.get_legend_handles_labels()

    ax.legend(
        handles=[legenda_nao_fraude, legenda_fraude, handles[0], handles[1], legenda_corte_vertical],
        title="Legenda",
        loc="best",
        frameon=True
    )

    plt.tight_layout()
    return fig


# ============================================================
# PROCESSAMENTO POR RANK
# ============================================================

def processar_rank_1x1(rank, df, scores_1x1, target_name, pasta_latex):
    colunas_necessarias = ["Feature", "Posicao_Rank", "AUC_PR", "MCC", "Score_Final", "Melhor_Ponto_Corte"]

    for coluna in colunas_necessarias:
        if coluna not in scores_1x1.columns:
            raise ValueError(f"O arquivo de scores precisa ter a coluna '{coluna}'.")

    if "Ponto_Corte_Medio" not in scores_1x1.columns:
        scores_1x1["Ponto_Corte_Medio"] = 0.5

    if rank not in scores_1x1["Posicao_Rank"].values:
        raise ValueError(f"Rank {rank} não encontrado. Ranks disponíveis: 1 até {scores_1x1['Posicao_Rank'].max()}.")

    linha_rank = scores_1x1.loc[scores_1x1["Posicao_Rank"] == rank].iloc[0]

    feature = linha_rank["Feature"]
    melhor_ponto_corte = float(linha_rank["Melhor_Ponto_Corte"])
    ponto_corte_medio = float(linha_rank["Ponto_Corte_Medio"])
    auc_pr = float(linha_rank["AUC_PR"])
    mcc = float(linha_rank["MCC"])

    if "Log_Loss_Norm" in linha_rank.index:
        log_loss_norm = float(linha_rank["Log_Loss_Norm"])
    else:
        log_loss_norm = 1 / (1 + float(linha_rank["Log_Loss"]))

    score_final = float(linha_rank["Score_Final"])

    if "Diferenca_Neg_Log_Veross" in linha_rank.index:
        diferenca_neg_log_veross = float(linha_rank["Diferenca_Neg_Log_Veross"])
    elif "Neg_Log_Veross_Com_Rotulo" in linha_rank.index and "Neg_Log_Veross_GMM" in linha_rank.index:
        diferenca_neg_log_veross = float(linha_rank["Neg_Log_Veross_Com_Rotulo"]) - float(linha_rank["Neg_Log_Veross_GMM"])
    else:
        diferenca_neg_log_veross = np.nan

    temp = df[[feature, target_name]].dropna().copy()
    X = temp[[feature]]
    y_real = temp[target_name].astype(int)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    gmm = GaussianMixture(
        n_components=2,
        covariance_type="full",
        random_state=42,
        n_init=3,
        reg_covar=1e-6
    )

    gmm.fit(X_scaled)

    clusters = gmm.predict(X_scaled)
    ct = pd.crosstab(clusters, y_real)

    if 1 not in ct.columns:
        raise ValueError("A classe fraude, valor 1, não foi encontrada no target.")

    cluster_fraude = ct[1].idxmax()

    responsabilidades = gmm.predict_proba(X_scaled)
    probabilidades = responsabilidades[:, cluster_fraude]
    probabilidades = np.clip(probabilidades, 1e-15, 1 - 1e-15)

    cm_melhor = gerar_matriz_confusao(y_real, probabilidades, melhor_ponto_corte)
    cm_medio = gerar_matriz_confusao(y_real, probabilidades, ponto_corte_medio)

    valores_melhor = preparar_valores_matriz(cm_melhor)
    valores_medio = preparar_valores_matriz(cm_medio)
    valores_ideal = preparar_valores_matriz_ideal(y_real)

    prefixo = f"rank_{rank}_{sanitizar_nome(feature)}"

    figs = {
        "boxplot": gerar_boxplot_1d_fig(temp, feature, target_name),
        "representacao_1d": gerar_representacao_1d_fig(temp, feature, target_name),
        "spearman": gerar_matriz_spearman_1d_fig(temp, feature, target_name),
        "responsabilidades_melhor_corte": gerar_responsabilidades_1d_fig(temp, feature, target_name, scaler, gmm, cluster_fraude, probabilidades, melhor_ponto_corte, "Responsabilidades GMM - Melhor Ponto de Corte"),
        "responsabilidades_corte_medio": gerar_responsabilidades_1d_fig(temp, feature, target_name, scaler, gmm, cluster_fraude, probabilidades, ponto_corte_medio, "Responsabilidades GMM - Ponto de Corte Médio"),
        "matriz_melhor_corte": gerar_fig_matriz_confusao(f"Rank {rank} - {feature} - Melhor Ponto de Corte", valores_melhor),
        "matriz_corte_medio": gerar_fig_matriz_confusao(f"Rank {rank} - {feature} - Ponto de Corte 0.5", valores_medio),
        "matriz_ideal": gerar_fig_matriz_confusao(f"Rank {rank} - {feature} - Matriz Ideal", valores_ideal),
    }

    imagens_base64 = {}

    for nome, fig in figs.items():
        nome_arquivo = f"{prefixo}_{nome}"
        salvar_figura(fig, Path(pasta_latex) / nome_arquivo)
        imagens_base64[nome] = fig_to_base64(fig)
        plt.close(fig)

    tabela_metricas_rank = pd.DataFrame([{
        "Rank": rank,
        "Feature": feature,
        "Melhor_Ponto_Corte": melhor_ponto_corte,
        "Ponto_Corte_Medio": ponto_corte_medio,
        "AUC_PR": auc_pr,
        "MCC": mcc,
        "Log_Loss_Norm": log_loss_norm,
        "Score_Final": score_final,
        "Diferenca_Neg_Log_Veross": diferenca_neg_log_veross,
        "GMM_N_Components": 2,
        "GMM_Covariance_Type": "full",
        "GMM_Random_State": 42,
        "GMM_N_Init": 3,
        "GMM_Reg_Covar": 1e-6,
        "Cluster_Fraude": int(cluster_fraude)
    }])

    tabela_matrizes_rank = pd.DataFrame([
        {
            "Rank": rank, "Feature": feature, "Cenario": "Melhor ponto de corte", "Threshold": melhor_ponto_corte,
            "TN": valores_melhor["raw"]["tn"], "FP": valores_melhor["raw"]["fp"], "FN": valores_melhor["raw"]["fn"], "TP": valores_melhor["raw"]["tp"],
            "FN_Pct_Real_Fraude": valores_melhor["fn"]["pct"], "TP_Pct_Real_Fraude": valores_melhor["tp"]["pct"],
            "TN_Pct_Real_Nao_Fraude": valores_melhor["tn"]["pct"], "FP_Pct_Real_Nao_Fraude": valores_melhor["fp"]["pct"]
        },
        {
            "Rank": rank, "Feature": feature, "Cenario": "Ponto de corte médio", "Threshold": ponto_corte_medio,
            "TN": valores_medio["raw"]["tn"], "FP": valores_medio["raw"]["fp"], "FN": valores_medio["raw"]["fn"], "TP": valores_medio["raw"]["tp"],
            "FN_Pct_Real_Fraude": valores_medio["fn"]["pct"], "TP_Pct_Real_Fraude": valores_medio["tp"]["pct"],
            "TN_Pct_Real_Nao_Fraude": valores_medio["tn"]["pct"], "FP_Pct_Real_Nao_Fraude": valores_medio["fp"]["pct"]
        },
        {
            "Rank": rank, "Feature": feature, "Cenario": "Ideal", "Threshold": np.nan,
            "TN": valores_ideal["raw"]["tn"], "FP": valores_ideal["raw"]["fp"], "FN": valores_ideal["raw"]["fn"], "TP": valores_ideal["raw"]["tp"],
            "FN_Pct_Real_Fraude": valores_ideal["fn"]["pct"], "TP_Pct_Real_Fraude": valores_ideal["tp"]["pct"],
            "TN_Pct_Real_Nao_Fraude": valores_ideal["tn"]["pct"], "FP_Pct_Real_Nao_Fraude": valores_ideal["fp"]["pct"]
        }
    ])

    tabela_metricas_html = tabela_metricas_rank.copy()
    for col in tabela_metricas_html.columns:
        if pd.api.types.is_float_dtype(tabela_metricas_html[col]):
            tabela_metricas_html[col] = tabela_metricas_html[col].apply(lambda x: formatar_float_html(x, 6))

    html_metricas = secao_tabela(
        titulo=f"Métricas do Rank {rank}",
        df=tabela_metricas_html,
        table_id=f"tabela_metricas_rank_{rank}",
        nome_csv=f"{prefixo}_metricas.csv"
    )

    html_melhor = html_matriz_visual(
        titulo=f"Matriz de Confusão (%) - {feature} - Melhor Ponto de Corte",
        valores=valores_melhor,
        imagem_base64=imagens_base64["matriz_melhor_corte"],
        nome_download=f"{prefixo}_matriz_melhor_corte.png"
    )

    html_medio = html_matriz_visual(
        titulo=f"Matriz de Confusão (%) - {feature} - Ponto de Corte 0.5",
        valores=valores_medio,
        imagem_base64=imagens_base64["matriz_corte_medio"],
        nome_download=f"{prefixo}_matriz_corte_medio.png"
    )

    html_ideal = html_matriz_visual(
        titulo="Matriz de Confusão Ideal (%)",
        valores=valores_ideal,
        imagem_base64=imagens_base64["matriz_ideal"],
        nome_download=f"{prefixo}_matriz_ideal.png",
        matriz_ideal=True
    )

    html_rank = f"""
    <section class="rank-section" id="rank-{rank}">
        <h1>Rank {rank} - Feature {html.escape(str(feature))}</h1>

        <div class="info-box">
            <div class="info-grid">

                <div class="info-item"><div class="info-label">Rank</div><div class="info-value">{rank}</div></div>
                <div class="info-item"><div class="info-label">Feature</div><div class="info-value">{html.escape(str(feature))}</div></div>
                <div class="info-item"><div class="info-label">Melhor Ponto de Corte</div><div class="info-value">{melhor_ponto_corte:.6f}</div></div>
                <div class="info-item"><div class="info-label">Ponto de Corte Médio</div><div class="info-value">{ponto_corte_medio:.6f}</div></div>
                <div class="info-item"><div class="info-label">AUC-PR</div><div class="info-value">{auc_pr:.6f}</div></div>
                <div class="info-item"><div class="info-label">MCC</div><div class="info-value">{mcc:.6f}</div></div>
                <div class="info-item"><div class="info-label">Log Loss Norm</div><div class="info-value">{log_loss_norm:.6f}</div></div>
                <div class="info-item"><div class="info-label">Score Final</div><div class="info-value">{score_final:.6f}</div></div>
                <div class="info-item"><div class="info-label">Diferença Neg. Log-Veross.</div><div class="info-value">{diferenca_neg_log_veross:.6f}</div></div>

            </div>
        </div>

        {html_metricas}
        {html_melhor}
        {html_medio}
        {html_ideal}

        {secao_imagem(f"Boxplot por Classe - {feature}", imagens_base64["boxplot"], f"{prefixo}_boxplot.png", f"Boxplot por Classe - {feature}")}
        {secao_imagem(f"Representação 1D com 100% dos Dados - {feature}", imagens_base64["representacao_1d"], f"{prefixo}_representacao_1d.png", f"Representação 1D - {feature}")}
        {secao_imagem(f"Correlação de Spearman entre Feature e Target - {feature} e Fraude", imagens_base64["spearman"], f"{prefixo}_spearman.png", f"Spearman - {feature}")}
        {secao_imagem(f"Responsabilidades Estimadas pela GMM - Melhor Ponto de Corte - {feature}", imagens_base64["responsabilidades_melhor_corte"], f"{prefixo}_responsabilidades_melhor_corte.png", f"Responsabilidades GMM melhor corte - {feature}")}
        {secao_imagem(f"Responsabilidades Estimadas pela GMM - Ponto de Corte Médio - {feature}", imagens_base64["responsabilidades_corte_medio"], f"{prefixo}_responsabilidades_corte_medio.png", f"Responsabilidades GMM corte médio - {feature}")}
    </section>
    """

    return {
        "rank": rank,
        "feature": feature,
        "html": html_rank,
        "tabela_metricas": tabela_metricas_rank,
        "tabela_matrizes": tabela_matrizes_rank
    }


# ============================================================
# FUNÇÃO PRINCIPAL UNIFICADA
# ============================================================

def gerar_relatorio_1x1_unificado(
    ranks=RANKS,
    arquivo_dados=ARQUIVO_DADOS,
    arquivo_scores=ARQUIVO_SCORES,
    pasta_saida=PASTA_SAIDA,
    nome_arquivo_html=ARQUIVO_HTML,
    pasta_latex=PASTA_LATEX,
    exportar_latex=True
):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_html = pasta_saida / nome_arquivo_html
    caminho_latex = pasta_saida / pasta_latex
    caminho_latex.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(arquivo_dados)
    scores_1x1 = pd.read_csv(arquivo_scores)

    if "status_fraude" in df.columns:
        target_name = "status_fraude"
    elif "Class" in df.columns:
        df = df.rename(columns={"Class": "status_fraude"})
        target_name = "status_fraude"
    else:
        raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")

    resultados = []

    for rank in ranks:
        resultados.append(
            processar_rank_1x1(
                rank=rank,
                df=df,
                scores_1x1=scores_1x1,
                target_name=target_name,
                pasta_latex=caminho_latex
            )
        )

    tabela_metricas_total = pd.concat([r["tabela_metricas"] for r in resultados], ignore_index=True)
    tabela_matrizes_total = pd.concat([r["tabela_matrizes"] for r in resultados], ignore_index=True)

    if exportar_latex:
        salvar_tabela_latex(
            tabela_metricas_total,
            caminho_latex / "tabela_metricas_ranks.tex",
            caption="Métricas dos melhores rankings do experimento 1x1.",
            label="tab:metricas-ranks-1x1",
            longtable=False
        )

        salvar_tabela_latex(
            tabela_matrizes_total,
            caminho_latex / "tabela_matrizes_confusao_ranks.tex",
            caption="Resultados das matrizes de confusão para os melhores rankings do experimento 1x1.",
            label="tab:matrizes-confusao-ranks-1x1",
            longtable=True
        )

        comandos_latex = r"""
% ============================================================
% COMANDOS LaTeX DE EXEMPLO
%
% Pacotes recomendados no preâmbulo:
% \usepackage{graphicx}
% \usepackage{float}
% \usepackage{booktabs}
% \usepackage{longtable}
% \usepackage{pdflscape}
% ============================================================

\begin{table}[H]
\centering
\caption{Métricas dos melhores rankings do experimento 1x1.}
\label{tab:metricas-ranks-1x1-main}
\input{1x1_visu_scores/tabela_metricas_ranks.tex}
\end{table}

\begin{landscape}
\small
\input{1x1_visu_scores/tabela_matrizes_confusao_ranks.tex}
\end{landscape}

% Exemplo de figura:
% \begin{figure}[H]
% \centering
% \includegraphics[width=\textwidth]{1x1_visu_scores/rank_1_FEATURE_representacao_1d.pdf}
% \caption{Representação 1D da feature selecionada no Rank 1.}
% \label{fig:rank1-representacao-1d}
% \end{figure}
"""
        (caminho_latex / "comandos_latex_exemplo.tex").write_text(comandos_latex, encoding="utf-8")

    tabela_metricas_total_html = tabela_metricas_total.copy()
    tabela_matrizes_total_html = tabela_matrizes_total.copy()

    for tabela in [tabela_metricas_total_html, tabela_matrizes_total_html]:
        for col in tabela.columns:
            if pd.api.types.is_float_dtype(tabela[col]):
                tabela[col] = tabela[col].apply(lambda x: formatar_float_html(x, 6))

    html_resumo_metricas = secao_tabela(
        "Tabela Geral de Métricas dos Ranks",
        tabela_metricas_total_html,
        "tabela_metricas_ranks",
        "tabela_metricas_ranks.csv"
    )

    html_resumo_matrizes = secao_tabela(
        "Tabela Geral das Matrizes de Confusão dos Ranks",
        tabela_matrizes_total_html,
        "tabela_matrizes_confusao_ranks",
        "tabela_matrizes_confusao_ranks.csv"
    )

    navegacao_links = "\n".join([
        f'<a href="#rank-{r["rank"]}">Rank {r["rank"]} - {html.escape(str(r["feature"]))}</a>'
        for r in resultados
    ])

    html_ranks = "\n".join([r["html"] for r in resultados])

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <title>Relatório 1x1 - Top Ranks</title>

        <style>
            body {{
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
                margin: 0;
                padding: 32px;
            }}

            .container {{
                max-width: 1250px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                margin-bottom: 28px;
                color: #020617;
            }}

            .main-title {{
                font-size: 34px;
                margin-bottom: 14px;
            }}

            .nav-box {{
                background: #ffffff;
                border-radius: 16px;
                padding: 18px;
                margin-bottom: 28px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
                display: flex;
                justify-content: center;
                gap: 12px;
                flex-wrap: wrap;
            }}

            .nav-box a {{
                text-decoration: none;
                background: #eff6ff;
                color: #1e3a8a;
                border: 1px solid #bfdbfe;
                padding: 8px 12px;
                border-radius: 999px;
                font-weight: 800;
                font-size: 13px;
            }}

            .rank-section {{
                margin-top: 46px;
                padding-top: 12px;
                border-top: 4px solid #cbd5e1;
            }}

            .info-box {{
                background: #ffffff;
                border-radius: 16px;
                padding: 20px 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .info-grid {{
                display: grid;
                grid-template-columns: repeat(3, 1fr);
                gap: 14px;
                margin-top: 14px;
            }}

            .info-item {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                padding: 12px 14px;
            }}

            .info-label {{
                font-size: 13px;
                font-weight: 700;
                color: #475569;
                margin-bottom: 6px;
            }}

            .info-value {{
                font-size: 18px;
                font-weight: 800;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
            }}

            .matrix-card,
            .card {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .matrix-card h2,
            .card h2 {{
                text-align: center;
                margin-top: 0;
                margin-bottom: 24px;
                color: #020617;
                font-size: 22px;
            }}

            .section-header {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 14px;
                flex-wrap: wrap;
                margin-bottom: 18px;
            }}

            .section-header h2 {{
                margin: 0;
            }}

            .download-btn {{
                border: 1px solid #bfdbfe;
                background: #eff6ff;
                color: #1e3a8a;
                padding: 8px 12px;
                border-radius: 10px;
                font-weight: 800;
                font-size: 13px;
                cursor: pointer;
                text-decoration: none;
                display: inline-block;
            }}

            .download-btn:hover {{
                background: #dbeafe;
            }}

            .table-wrapper {{
                overflow-x: auto;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                max-height: 580px;
                overflow-y: auto;
            }}

            .data-table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 13px;
                margin-top: 0;
            }}

            .data-table th {{
                background: #0f172a;
                color: white;
                padding: 10px 8px;
                text-align: left;
                position: sticky;
                top: 0;
                z-index: 1;
            }}

            .data-table td {{
                border-bottom: 1px solid #e2e8f0;
                padding: 8px;
                color: #020617;
                white-space: nowrap;
            }}

            .data-table tr:nth-child(even) {{
                background: #f8fafc;
            }}

            .matrix-area {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 34px;
            }}

            .matrix-wrapper {{
                display: grid;
                grid-template-columns: 180px 1fr 1fr;
                grid-template-rows: 48px 190px 190px;
                width: 950px;
            }}

            .corner {{
                background: transparent;
            }}

            .x-label {{
                display: flex;
                align-items: center;
                justify-content: center;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
                border-bottom: 1px solid #e5e7eb;
            }}

            .y-label {{
                display: flex;
                align-items: center;
                justify-content: flex-end;
                padding-right: 18px;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
            }}

            .cell {{
                display: flex;
                flex-direction: column;
                align-items: center;
                justify-content: center;
                min-height: 180px;
                border: 1px solid #e5e7eb;
                font-size: 20px;
                text-align: center;
                color: #020617 !important;
            }}

            .pct {{
                font-size: 30px;
                font-weight: 900;
                margin-bottom: 4px;
                color: #020617 !important;
            }}

            .count {{
                font-size: 24px;
                font-weight: 900;
                margin-bottom: 8px;
                color: #020617 !important;
            }}

            .cell-desc {{
                font-size: 13px;
                font-weight: 700;
                opacity: 1;
                color: #020617 !important;
            }}

            .q95 {{ background: #08306b; }}
            .q85 {{ background: #08519c; }}
            .q70 {{ background: #2171b5; }}
            .q50 {{ background: #6baed6; }}
            .q30 {{ background: #c6dbef; }}
            .q10 {{ background: #eff6ff; }}

            .legend {{
                position: relative;
                display: flex;
                flex-direction: column;
                align-items: center;
                min-width: 115px;
            }}

            .legend-title {{
                font-weight: 800;
                font-size: 15px;
                margin-bottom: 10px;
                color: #020617;
            }}

            .colorbar {{
                width: 30px;
                height: 310px;
                border-radius: 16px;
                background: linear-gradient(to bottom, #08306b 0%, #08519c 18%, #2171b5 36%, #6baed6 58%, #c6dbef 78%, #eff6ff 100%);
                border: 1px solid #cbd5e1;
            }}

            .legend-label-top {{
                position: absolute;
                top: 43px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .legend-label-bottom {{
                position: absolute;
                top: 335px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .plot-img {{
                display: block;
                max-width: 100%;
                margin: 0 auto;
                border-radius: 12px;
                border: 1px solid #e2e8f0;
            }}

            @media (max-width: 1100px) {{
                .info-grid {{
                    grid-template-columns: repeat(2, 1fr);
                }}

                .matrix-area {{
                    flex-direction: column;
                }}

                .matrix-wrapper {{
                    width: 100%;
                    grid-template-columns: 150px 1fr 1fr;
                }}
            }}

            @media (max-width: 700px) {{
                body {{
                    padding: 16px;
                }}

                .info-grid {{
                    grid-template-columns: 1fr;
                }}

                .matrix-wrapper {{
                    grid-template-columns: 120px 1fr 1fr;
                    grid-template-rows: 48px 160px 160px;
                }}

                .pct {{ font-size: 22px; }}
                .count {{ font-size: 18px; }}

                .y-label,
                .x-label {{
                    font-size: 14px;
                }}
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1 class="main-title">Relatório Unificado 1x1 - Top Ranks</h1>

            <div class="nav-box">
                {navegacao_links}
            </div>

            {html_resumo_metricas}

            {html_resumo_matrizes}

            {html_ranks}

        </div>

        <script>
            function limparTextoCSV(texto) {{
                if (texto === null || texto === undefined) {{
                    return "";
                }}

                texto = String(texto).replace(/\\n/g, " ").replace(/\\s+/g, " ").trim();

                if (texto.includes(";") || texto.includes('"')) {{
                    texto = '"' + texto.replace(/"/g, '""') + '"';
                }}

                return texto;
            }}

            function baixarTabelaCSV(tableId, filename) {{
                const tabela = document.getElementById(tableId);

                if (!tabela) {{
                    alert("Tabela não encontrada: " + tableId);
                    return;
                }}

                const linhas = [];

                tabela.querySelectorAll("tr").forEach(function(row) {{
                    const celulas = Array.from(row.querySelectorAll("th, td"));
                    const linha = celulas.map(celula => limparTextoCSV(celula.innerText)).join(";");
                    linhas.push(linha);
                }});

                const csv = "\\ufeff" + linhas.join("\\n");
                const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
                const url = URL.createObjectURL(blob);

                const link = document.createElement("a");
                link.href = url;
                link.download = filename;
                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);

                URL.revokeObjectURL(url);
            }}
        </script>
    </body>
    </html>
    """

    caminho_html.write_text(html_final, encoding="utf-8")

    print("=" * 80)
    print("RELATÓRIO 1x1 UNIFICADO GERADO COM SUCESSO")
    print("=" * 80)
    print(f"HTML salvo em: {caminho_html.resolve()}")
    print(f"Arquivos LaTeX/imagens salvos em: {caminho_latex.resolve()}")
    print("=" * 80)

    return {
        "caminho_html": caminho_html,
        "caminho_latex": caminho_latex,
        "tabela_metricas": tabela_metricas_total,
        "tabela_matrizes": tabela_matrizes_total
    }


# ============================================================
# CHAMADA
# ============================================================

resultado_1x1_visu_scores = gerar_relatorio_1x1_unificado(
    ranks=(1, 2, 3),
    arquivo_dados="creditcard.csv",
    arquivo_scores="1x1_scores.csv",
    pasta_saida=".",
    nome_arquivo_html="1x1_visu_scores.html",
    pasta_latex="1x1_visu_scores",
    exportar_latex=True
)

resultado_1x1_visu_scores["caminho_html"]


C:\Users\Vitor Craveiro\AppData\Local\Temp\ipykernel_23160\560908761.py:358: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  box = ax.boxplot(
C:\Users\Vitor Craveiro\AppData\Local\Temp\ipykernel_23160\560908761.py:358: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  box = ax.boxplot(
C:\Users\Vitor Craveiro\AppData\Local\Temp\ipykernel_23160\560908761.py:358: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  box = ax.boxplot(


RELATÓRIO 1x1 UNIFICADO GERADO COM SUCESSO
HTML salvo em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1x1_visu_scores.html
Arquivos LaTeX/imagens salvos em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1x1_visu_scores


WindowsPath('1x1_visu_scores.html')